In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
transactions = pd.read_csv("../data/transactions.csv")
budgets = pd.read_csv("../data/budgets.csv")
categories = pd.read_csv("../data/categories.csv")

print("Datasets loaded successfully!")

print("Transactions:", transactions.shape)
print("Budgets:", budgets.shape)
print("Categories:", categories.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/transactions.csv'

In [ ]:
import os

print("Current notebook directory:", os.getcwd())
print("\nFiles and folders here:")
print(os.listdir())



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
transactions = pd.read_csv("Data/transactions.csv")
budgets = pd.read_csv("Data/budgets.csv")
categories = pd.read_csv("Data/categories.csv")

print("All datasets loaded successfully!")

print("Transactions:", transactions.shape)
print("Budgets:", budgets.shape)
print("Categories:", categories.shape)

In [ ]:
# Display the first 5 rows of each dataset

print("TRANSACTIONS DATASET")
display(transactions.head())

print("BUDGETS DATASET")
display(budgets.head())

print("CATEGORIES DATASET")
display(categories.head())

In [ ]:
print("TRANSACTIONS")
transactions.info()

print("\nBUDGETS")
budgets.info()

print("\nCATEGORIES")
categories.info()

In [ ]:
# Check missing values
print("MISSING VALUES")
print("\nTransactions:")
print(transactions.isnull().sum())

print("\nBudgets:")
print(budgets.isnull().sum())

print("\nCategories:")
print(categories.isnull().sum())


# Check duplicate rows
print("\nDUPLICATE ROWS")
print("Transactions:", transactions.duplicated().sum())
print("Budgets:", budgets.duplicated().sum())
print("Categories:", categories.duplicated().sum())


# Check column names
print("\nCOLUMN NAMES")
print("Transactions:", transactions.columns.tolist())
print("Budgets:", budgets.columns.tolist())
print("Categories:", categories.columns.tolist())

In [ ]:
# Convert date columns to datetime format
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"], errors="coerce"
)

budgets["start_date"] = pd.to_datetime(
    budgets["start_date"], errors="coerce"
)

budgets["end_date"] = pd.to_datetime(
    budgets["end_date"], errors="coerce"
)

# Check invalid dates
print("Invalid transaction dates:", transactions["transaction_date"].isna().sum())
print("Invalid budget start dates:", budgets["start_date"].isna().sum())
print("Invalid budget end dates:", budgets["end_date"].isna().sum())

# Check financial amounts
print("\nNegative transaction amounts:", (transactions["amount"] < 0).sum())
print("Zero transaction amounts:", (transactions["amount"] == 0).sum())
print("Negative budget amounts:", (budgets["budget_amount"] < 0).sum())

# Check budget date ranges
print(
    "Budgets with start date after end date:",
    (budgets["start_date"] > budgets["end_date"]).sum()
)

In [ ]:
# Display budget records where the start date is after the end date
invalid_budgets = budgets[
    budgets["start_date"] > budgets["end_date"]
]

print("Number of affected records:", len(invalid_budgets))

invalid_budgets[
    ["id", "user_id", "category_id", "budget_amount",
     "start_date", "end_date"]
].head(10)

In [ ]:
# Flag budgets with invalid date ranges
budgets["invalid_date_range"] = (
    budgets["start_date"] > budgets["end_date"]
)

print("Invalid budget date ranges:", budgets["invalid_date_range"].sum())
print("Valid budget date ranges:", (~budgets["invalid_date_range"]).sum())

In [ ]:
# Inspect budgets with missing amounts
missing_budgets = budgets[budgets["budget_amount"].isna()]

print("Missing budget amounts:", len(missing_budgets))

missing_budgets[
    ["id", "user_id", "category_id", "budget_amount",
     "start_date", "end_date"]
].head(10)

In [ ]:
# Check missing values before cleaning
print("Missing values before cleaning:")

print("\nTransactions:")
print(transactions.isnull().sum())

print("\nBudgets:")
print(budgets.isnull().sum())

print("\nCategories:")
print(categories.isnull().sum())

# Create working copies so the original dataframes remain unchanged
transactions_clean = transactions.copy()
budgets_clean = budgets.copy()
categories_clean = categories.copy()

print("\nWorking copies created successfully.")

In [ ]:
# Fill missing transaction notes
transactions_clean["notes"] = (
    transactions_clean["notes"].fillna("Not provided")
)

# Keep missing budget amounts as NaN
# Flag invalid budget date ranges
budgets_clean["invalid_date_range"] = (
    budgets_clean["start_date"] > budgets_clean["end_date"]
)

# Verify the changes
print("Missing transaction notes:",
      transactions_clean["notes"].isna().sum())

print("Missing budget amounts:",
      budgets_clean["budget_amount"].isna().sum())

print("Invalid budget date ranges:",
      budgets_clean["invalid_date_range"].sum())

In [ ]:
# Get valid category IDs
valid_category_ids = set(categories_clean["id"])

# Check transactions referencing missing categories
invalid_transaction_categories = (
    ~transactions_clean["category_id"].isin(valid_category_ids)
).sum()

# Check budgets referencing missing categories
invalid_budget_categories = (
    ~budgets_clean["category_id"].isin(valid_category_ids)
).sum()

print("Transactions with invalid category IDs:",
      invalid_transaction_categories)

print("Budgets with invalid category IDs:",
      invalid_budget_categories)

In [ ]:
# Inspect category names
print("Category names:")
print(categories_clean["name"].sort_values().to_list())

print("\nDuplicate category names:",
      categories_clean["name"].duplicated().sum())

print("\nUnique expense types:")
print(categories_clean["expense_type"].unique())

In [ ]:
# Display duplicate category names
duplicate_categories = categories_clean[
    categories_clean["name"].duplicated(keep=False)
].sort_values("name")

duplicate_categories

In [ ]:
# Count categories by expense type
print("Expense type distribution:")
print(categories_clean["expense_type"].value_counts())

# Summarize average amounts by expense type
print("\nAverage amount by expense type:")
print(
    categories_clean.groupby("expense_type")["average_amount"]
    .agg(["count", "mean", "min", "max"])
    .round(2)
)

In [ ]:
# Join transactions with category information
transactions_analysis = transactions_clean.merge(
    categories_clean[["id", "name", "expense_type"]],
    left_on="category_id",
    right_on="id",
    how="left",
    suffixes=("", "_category")
)

# Summarize spending by category
category_spending = (
    transactions_analysis
    .groupby("name")["amount"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
    .round(2)
)

category_spending.columns = [
    "Total Spending", "Average Transaction", "Transaction Count"
]

category_spending

In [ ]:
# Plot total spending by category
category_spending["Total Spending"].sort_values().plot(
    kind="barh",
    figsize=(10, 6),
    title="Total Spending by Category"
)

plt.xlabel("Total Spending")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
# Extract the transaction month
transactions_analysis["month"] = (
    transactions_analysis["transaction_date"].dt.to_period("M")
)

# Calculate monthly spending
monthly_spending = (
    transactions_analysis
    .groupby("month")["amount"]
    .sum()
)

# Plot monthly spending
monthly_spending.plot(
    kind="line",
    figsize=(12, 5),
    marker="o",
    title="Monthly Spending Trend"
)

plt.xlabel("Month")
plt.ylabel("Total Spending")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Merge transactions with expense types
expense_type_analysis = transactions_clean.merge(
    categories_clean[["id", "expense_type"]],
    left_on="category_id",
    right_on="id",
    how="left",
    suffixes=("", "_category")
)

# Calculate spending by expense type
type_spending = (
    expense_type_analysis
    .groupby("expense_type")["amount"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
    .round(2)
)

type_spending.columns = [
    "Total Spending",
    "Average Transaction",
    "Transaction Count"
]

type_spending

In [ ]:
type_spending["Total Spending"].sort_values().plot(
    kind="bar",
    figsize=(8, 5),
    title="Total Spending by Expense Type"
)

plt.xlabel("Expense Type")
plt.ylabel("Total Spending")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Count valid and invalid budget records
print("Total budget records:", len(budgets_clean))

print(
    "Valid date ranges:",
    (~budgets_clean["invalid_date_range"]).sum()
)

print(
    "Missing budget amounts:",
    budgets_clean["budget_amount"].isna().sum()
)

print(
    "Records with valid dates and amounts:",
    (
        (~budgets_clean["invalid_date_range"])
        & budgets_clean["budget_amount"].notna()
    ).sum()
)

In [ ]:
# Keep budgets with valid date ranges and amounts
valid_budgets = budgets_clean[
    (~budgets_clean["invalid_date_range"])
    & budgets_clean["budget_amount"].notna()
].copy()

# Confirm the cleaned dataset
print("Usable budget records:", len(valid_budgets))

print("\nDate range:")
print("Earliest start:", valid_budgets["start_date"].min())
print("Latest end:", valid_budgets["end_date"].max())

print("\nMissing budget amounts:", valid_budgets["budget_amount"].isna().sum())
print(
    "Invalid date ranges:",
    (valid_budgets["start_date"] > valid_budgets["end_date"]).sum()
)

In [ ]:
# Match transactions to budgets by user and category
budget_comparison = valid_budgets.merge(
    transactions_clean,
    on=["user_id", "category_id"],
    how="left",
    suffixes=("_budget", "_transaction")
)

# Keep transactions within each budget's date range
budget_comparison = budget_comparison[
    (budget_comparison["transaction_date"] >= budget_comparison["start_date"])
    & (budget_comparison["transaction_date"] <= budget_comparison["end_date"])
]

# Calculate spending against each budget
budget_summary = (
    budget_comparison
    .groupby(["id_budget", "user_id", "category_id", "budget_amount"])
    .agg(actual_spending=("amount", "sum"))
    .reset_index()
)

budget_summary["budget_difference"] = (
    budget_summary["budget_amount"] - budget_summary["actual_spending"]
)

budget_summary["status"] = np.where(
    budget_summary["actual_spending"] > budget_summary["budget_amount"],
    "Over Budget",
    "Within Budget"
)

budget_summary.head(10)

In [ ]:
print("Category IDs:")
print(sorted(categories_clean["id"].unique()))

print("\nBudget category IDs:")
print(sorted(valid_budgets["category_id"].unique()))

print("\nTransaction category IDs:")
print(sorted(transactions_clean["category_id"].unique()))

In [ ]:
print("Budgets compared:", len(budget_summary))

print("\nBudget status:")
print(budget_summary["status"].value_counts())

print("\nTotal budget amount:",
      round(budget_summary["budget_amount"].sum(), 2))

print("Total actual spending:",
      round(budget_summary["actual_spending"].sum(), 2))

print("Total budget remaining:",
      round(budget_summary["budget_difference"].sum(), 2))

In [ ]:
# Count transactions matching each valid budget's user and category
budget_matches = valid_budgets.merge(
    transactions_clean,
    on=["user_id", "category_id"],
    how="left",
    suffixes=("_budget", "_transaction")
)

# Count matches before applying date filters
matched_pairs = budget_matches[
    budget_matches["transaction_date"].notna()
]

print("Valid budgets:", len(valid_budgets))
print("Budgets with matching user/category transactions:",
      matched_pairs[["id_budget", "user_id", "category_id"]]
      .drop_duplicates().shape[0])

In [ ]:
# Keep transactions within each budget's date range
in_period = budget_matches[
    budget_matches["transaction_date"].notna()
    & (budget_matches["transaction_date"] >= budget_matches["start_date"])
    & (budget_matches["transaction_date"] <= budget_matches["end_date"])
]

# Calculate spending for matching budget periods
period_spending = (
    in_period.groupby("id_budget")["amount"]
    .sum()
)

# Add spending back to every valid budget
budget_summary_all = valid_budgets.copy()

budget_summary_all["actual_spending"] = (
    budget_summary_all["id"].map(period_spending).fillna(0)
)

budget_summary_all["budget_difference"] = (
    budget_summary_all["budget_amount"]
    - budget_summary_all["actual_spending"]
)

budget_summary_all["status"] = np.where(
    budget_summary_all["actual_spending"]
    > budget_summary_all["budget_amount"],
    "Over Budget",
    "Within Budget"
)

print("Budgets analyzed:", len(budget_summary_all))
print("\nBudget status:")
print(budget_summary_all["status"].value_counts())

In [ ]:
budget_status_counts = budget_summary_all["status"].value_counts()

budget_status_counts.plot(
    kind="bar",
    figsize=(7, 4),
    title="Budget Status Overview"
)

plt.xlabel("Budget Status")
plt.ylabel("Number of Budgets")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
total_spending = transactions_clean["amount"].sum()
average_transaction = transactions_clean["amount"].mean()
highest_transaction = transactions_clean["amount"].max()
lowest_transaction = transactions_clean["amount"].min()

print(f"Total Spending: ₹{total_spending:,.2f}")
print(f"Average Transaction: ₹{average_transaction:,.2f}")
print(f"Highest Transaction: ₹{highest_transaction:,.2f}")
print(f"Lowest Transaction: ₹{lowest_transaction:,.2f}")
print(f"Total Transactions: {len(transactions_clean)}")
print(f"Budgets Over Budget: {(budget_summary_all['status'] == 'Over Budget').sum()}")

In [ ]:
# Key Financial Insights

total_spending = transactions_clean["amount"].sum()
average_transaction = transactions_clean["amount"].mean()
total_transactions = len(transactions_clean)

top_category = category_spending["Total Spending"].idxmax()
top_category_spending = category_spending["Total Spending"].max()

highest_month = monthly_spending.idxmax()
lowest_month = monthly_spending.idxmin()

fixed_spending = type_spending.loc["fixed", "Total Spending"]
variable_spending = type_spending.loc["variable", "Total Spending"]
discretionary_spending = type_spending.loc["discretionary", "Total Spending"]

print("========== FINANCIAL INSIGHTS ==========")
print(f"Total Spending: ₹{total_spending:,.2f}")
print(f"Total Transactions: {total_transactions}")
print(f"Average Transaction: ₹{average_transaction:,.2f}")
print(f"Highest Spending Category: {top_category}")
print(f"Spending in Top Category: ₹{top_category_spending:,.2f}")
print(f"Highest Spending Month: {highest_month}")
print(f"Lowest Spending Month: {lowest_month}")
print(f"Fixed Expenses: ₹{fixed_spending:,.2f}")
print(f"Variable Expenses: ₹{variable_spending:,.2f}")
print(f"Discretionary Expenses: ₹{discretionary_spending:,.2f}")

In [ ]:
import os

os.makedirs("Reports", exist_ok=True)

transactions_clean.to_csv(
    "Reports/cleaned_transactions.csv", index=False
)

budgets_clean.to_csv(
    "Reports/cleaned_budgets.csv", index=False
)

categories_clean.to_csv(
    "Reports/cleaned_categories.csv", index=False
)

print("Cleaned datasets exported successfully!")
print("Location: Reports/")

In [64]:
report = f"""
# Financial Data Analysis Report

## 1. Project Overview
This project analyzes financial transactions, budgets, and expense categories
to identify spending patterns and evaluate budget performance.

## 2. Dataset Summary
- Total transactions: {len(transactions_clean)}
- Total budget records: {len(budgets_clean)}
- Total expense categories: {len(categories_clean)}

## 3. Key Financial Metrics
- Total spending: ₹{transactions_clean['amount'].sum():,.2f}
- Average transaction: ₹{transactions_clean['amount'].mean():,.2f}
- Highest transaction: ₹{transactions_clean['amount'].max():,.2f}
- Lowest transaction: ₹{transactions_clean['amount'].min():,.2f}

## 4. Key Findings
- Highest-spending category: {top_category}
- Spending in that category: ₹{top_category_spending:,.2f}
- Highest-spending month: {highest_month}
- Lowest-spending month: {lowest_month}
- Fixed expenses: ₹{fixed_spending:,.2f}
- Variable expenses: ₹{variable_spending:,.2f}
- Discretionary expenses: ₹{discretionary_spending:,.2f}

## 5. Budget Analysis
- Valid budgets analyzed: {len(budget_summary_all)}
- Within budget: {(budget_summary_all['status'] == 'Within Budget').sum()}
- Over budget: {(budget_summary_all['status'] == 'Over Budget').sum()}

Note: Budgets without matching transactions during their date ranges were
treated as having zero observed spending. Budget results should therefore
be interpreted cautiously.

## 6. Conclusion
The analysis identifies spending patterns across categories, months, and
expense types. These findings can support better budgeting and financial
planning. Further analysis should investigate missing budget amounts and
inconsistent budget date ranges.
"""

with open("Reports/Financial_Analysis_Report.md", "w", encoding="utf-8") as file:
    file.write(report)

print("Financial analysis report created successfully!")

Financial analysis report created successfully!
